# Configuration A — Unified Direct-LLM Baseline

This local replacement preserves the supplied baseline's experimental condition:

`ULB + Sparkov model evidence → direct LLM → investigation answer`

It uses no policy retrieval and no programmatic guardrail enforcement. Unlike the
original Colab prototype, it consumes the leakage-safe outputs of both current model
notebooks and keeps their incompatible feature spaces source-labelled and nested.



In [1]:
from __future__ import annotations

from pathlib import Path
import json
import os
import platform
import sys


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "evidence.py").exists():
            return candidate
    raise FileNotFoundError("Run from thesis_code or one of its subdirectories.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "configuration_a"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))

from src.evidence import load_unified_evidence  # noqa: E402
from src.llm_backend import Configuration, prepare_request  # noqa: E402
from src.model_evidence_retrieval import scope_evidence_for_question  # noqa: E402

print({"python": platform.python_version(), "project_root": str(PROJECT_ROOT)})



{'python': '3.13.5', 'project_root': '<REPOSITORY_ROOT>'}


## 1. Load the combined model evidence

First run both modelling notebooks, then run:

```bash
python3 tools/build_unified_evidence.py
```

Offline `actual_class` values are deliberately absent from this LLM context.



In [2]:
unified_evidence = load_unified_evidence(PROJECT_ROOT)
print("Evidence schema:", unified_evidence["schema_version"])
for dataset in unified_evidence["datasets"]:
    print({
        "dataset": dataset["dataset_id"],
        "selected_model": dataset["model_name"],
        "total_alerts": dataset["alert_count"],
        "alerts_in_context": len(dataset["top_alerts"]),
    })



Evidence schema: 1.0
{'dataset': 'ULB', 'selected_model': 'Random forest', 'total_alerts': 86, 'alerts_in_context': 25}
{'dataset': 'Sparkov', 'selected_model': 'Random forest', 'total_alerts': 2384, 'alerts_in_context': 25}


## 2. Bound the direct prompt size

Configuration A still receives evidence from both datasets, but only the highest-score
alerts are included. This is deterministic score ordering, not retrieval.



In [3]:
QUESTION = (
    "Considering the ULB and Sparkov model outputs together, summarize the highest-risk "
    "alerts, explain what can and cannot be inferred, and recommend investigation priorities."
)

prompt_evidence, evidence_routing = scope_evidence_for_question(
    unified_evidence,
    QUESTION,
    historical_records_per_dataset=1,
    alerts_per_dataset=2,
)
print("Question-aware evidence routing:", evidence_routing)

prepared_request = prepare_request(
    configuration=Configuration.A,
    question=QUESTION,
    unified_evidence=prompt_evidence,
)
print("Request ID:", prepared_request.request_id)
print("Evidence records:", len(prepared_request.evidence_ids))
print(prepared_request.prompt[:4_000])



Question-aware evidence routing: {'question': 'Considering the ULB and Sparkov model outputs together, summarize the highest-risk alerts, explain what can and cannot be inferred, and recommend investigation priorities.', 'historical_records_per_dataset': 1, 'alerts_per_dataset': 2, 'included_alerts': True, 'included_metrics': False, 'evidence_ids': ['historical:ulb:relative-time', 'historical:sparkov:hour', 'model-output:ulb:ulb-test-42588', 'model-output:ulb:ulb-test-15683', 'model-output:sparkov:sparkov-test-355667', 'model-output:sparkov:sparkov-test-113754']}
Request ID: dc5c8699-5268-4c08-b1b9-bfb5cf0b0674
Evidence records: 6
{
  "instruction": "Answer directly from the supplied fraud-model evidence. This is the baseline condition: do not retrieve policies or add external context. Explain dataset limitations and never invent meanings for ULB V1-V28.",
  "question": "Considering the ULB and Sparkov model outputs together, summarize the highest-risk alerts, explain what can and cann

## 3. Optional local Qwen baseline

Install the optional packages in `requirements-llm.txt`. Model weights are downloaded
from Hugging Face on the first run. Set `RUN_LOCAL_LLM = True` only when ready.



In [4]:
LLM_MODEL = "Qwen/Qwen3-0.6B"
RUN_LOCAL_LLM = os.getenv("RUN_LOCAL_LLM", "false").lower() == "true"


def load_local_model(model_name: str):
    try:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
    except ImportError as error:
        raise RuntimeError(
            "Install optional packages with: "
            "python3 -m pip install -r requirements-llm.txt"
        ) from error

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype="auto",
    )
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    model.to(device)
    model.eval()
    return tokenizer, model, torch


def generate_direct_response(prompt: str, tokenizer, model, torch) -> str:
    """Direct generation matching the supplied Configuration A design."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are a credit-card fraud investigation assistant. Analyze only "
                "the model evidence in the user prompt and do not invent unavailable "
                "facts or meanings for anonymized features."
            ),
        },
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    )
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    with torch.no_grad():
        torch.manual_seed(42)
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.2,
            do_sample=True,
            top_p=0.9,
        )
    generated_tokens = outputs[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()



## 4. Execute and save Configuration A



In [5]:
if RUN_LOCAL_LLM:
    tokenizer, model, torch = load_local_model(LLM_MODEL)
    llm_answer = generate_direct_response(
        prepared_request.prompt,
        tokenizer,
        model,
        torch,
    )
else:
    llm_answer = "NOT_RUN: set RUN_LOCAL_LLM=True after installing optional dependencies."

result = {
    "request_id": prepared_request.request_id,
    "configuration": prepared_request.configuration,
    "question": prepared_request.question,
    "evidence_ids": prepared_request.evidence_ids,
    "llm_model": LLM_MODEL,
    "llm_answer": llm_answer,
    "retrieval_used": False,
    "guardrail_enforcement_used": False,
}

output_path = ARTIFACT_DIR / "configuration_a_result.json"
output_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
print(llm_answer)
print("Saved:", output_path)



/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|                                  | 0/311 [00:00<?, ?it/s]

Loading weights: 100%|██████████████████████| 311/311 [00:00<00:00, 4014.61it/s]

Considering the ULB and Sparkov model outputs together, the highest-risk alerts are NodeList 1 and 2, which indicate fraud probabilities of 0.999994 and 0.999971, respectively. These alerts are based on the ULB dataset, which provides real anonymized benchmark data, and the Sparkov dataset, which contains synthetic behavioural data. 

From the ULB dataset, the highest observed fraud rate relative to hours is at 26 hours with a fraud rate of 0.01414677, and the Sparkov dataset shows a fraud rate of 0.02834167 at 22 hours. 

The ULB dataset supports the interpretation of real-bank behavior, while the Sparkov dataset provides insights into synthetic patterns. However, the ULB dataset does not support claims of independent real-bank deployment validity, and the Sparkov dataset does not provide evidence of real-bank deployment. 

Investigation priorities should include reviewing the ULB dataset for potential anomalies and verifying the Sparkov dataset for patterns that could indicate real-b

## Interpretation boundary

Configuration A intentionally lacks retrieval grounding and enforcement. Its output
is an experimental baseline, not an operational fraud decision. Compare it with B and
C using the same question, alert evidence, LLM version, decoding parameters, and run
conditions.
